# 00_setup — one-time workspace setup

Creates the Unity Catalog schemas and volume the project uses, then smoke-tests the libraries and the World Bank API.
Safe to re-run: every step is idempotent.

**Attach to:** `wb-ml-single` (18 LTS ML, single node). Not Serverless.

## 1. Schemas and volume (medallion layout)

**What:** creates `bronze` (raw data as ingested), `silver` (cleaned, leakage-safe), `gold` (model-ready features), `ml` (registered models), and the `bronze.raw_docs` volume for raw files (API JSON snapshots, PDFs).

**Why:** separating layers means a bug in silver is fixed by re-running silver from bronze — no API calls needed. `IF NOT EXISTS` makes re-runs harmless; `COMMENT`s document the catalog for anyone browsing it.

In [1]:
%sql
USE CATALOG dbw_wb_outcome;

CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Raw data as ingested from World Bank APIs and documents';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Cleaned, standardized, leakage-safe data';
CREATE SCHEMA IF NOT EXISTS gold   COMMENT 'Model-ready feature tables keyed by project_id';
CREATE SCHEMA IF NOT EXISTS ml     COMMENT 'Registered ML models';

CREATE VOLUME IF NOT EXISTS bronze.raw_docs COMMENT 'Raw PAD/PID PDFs and source files';

## 2. Install the three libraries the ML runtime doesn't include

**What:** installs `wbgapi` (World Bank Indicators API client), `pymupdf` (PDF text extraction), `faiss-cpu` (vector search for the optional RAG step), then restarts Python so they can be imported.

**Why:** 18 LTS ML already ships xgboost, MLflow, optuna, shap, and sentence-transformers. These installs are notebook-scoped and vanish when the cluster restarts — later the job definition will install them automatically.

In [2]:
%pip install wbgapi pymupdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [3]:
dbutils.library.restartPython()

## 3. Smoke test

**What:** imports every library the pipeline needs, calls the World Bank API for Kenya, and writes/reads/deletes a test file in the volume.

**Why:** proves the runtime, internet access, and volume permissions all work *before* writing real pipeline code. Expect xgboost 3.1.x and mlflow 3.8.x. The CUDA/TensorFlow log lines are harmless (no GPU on this cluster).

In [4]:
import xgboost, mlflow, optuna, shap, sentence_transformers
import wbgapi as wb, pymupdf, faiss

print("xgboost", xgboost.__version__, "| mlflow", mlflow.__version__)
print("sentence-transformers", sentence_transformers.__version__)

print(wb.economy.get("KEN"))

path = "/Volumes/dbw_wb_outcome/bronze/raw_docs/_smoke_test.txt"
with open(path, "w") as f:
    f.write("hello volume")
print(open(path).read())
dbutils.fs.rm(path)

2026-10-16 15:54:47.828326: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-16 15:54:47.853551: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-16 15:54:48.593363: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-16 15:54:56.788769: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

xgboost 3.1.1 | mlflow 3.8.1
sentence-transformers 5.1.2
{'id': 'KEN', 'value': 'Kenya', 'aggregate': False, 'longitude': 36.8126, 'latitude': -1.27975, 'region': 'SSF', 'adminregion': 'SSA', 'lendingType': 'IDB', 'incomeLevel': 'LMC', 'capitalCity': 'Nairobi'}
hello volume


True